In [ ]:
#stock  market agent sample 1
#Jack Baxter 

In [22]:
from langgraph.checkpoint.memory import MemorySaver
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import SystemMessage
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from datetime import datetime, timedelta 
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import getpass
import os
import requests
import time
import json

In [3]:
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_API_KEY'] = getpass.getpass('Enter the API key for LangSmith: ')
os.environ['TAVILY_API_KEY'] = getpass.getpass('Enter the API key for Tavily: ')
os.environ['XAI_API_KEY'] = getpass.getpass('Enter API key for xAI: ')
os.environ['FINNHUB_API_KEY'] = getpass.getpass('Enter API key for FinnHub: ')
os.environ['ALPHAVANTAGE_API_KEY'] = getpass.getpass('Enter API key for Alpha Vantage: ')
os.environ['FRED_API_KEY'] = getpass.getpass('Enter API key for FRED API: ')

In [4]:
#create all necessary tools here: 
#downloads, installs, imports

In [5]:
#create Tavily search tool (general)
search = TavilySearch(max_results=10)

In [23]:
@tool
def fetch_market_snapshot():
    #relevant US market info fetch
    """
    Fetches a real-time snapshot of relevant US stock market information focused on day trading metrics.
    Includes top gainers/losers/active stocks, major indices, VIX, and enhances top movers with technical indicators like RSI and MACD.
    Also includes recent market news sentiment. Assumes ALPHAVANTAGE_API_KEY is set in environment.
    Returns:
        dict: A dictionary containing market snapshot data.
    """
    snapshot = {}
    alpha_vantage_key = os.environ['ALPHAVANTAGE_API_KEY']
    fred_key = os.environ['FRED_API_KEY']
    #Fetch top gainers, losers, and most active stocks
    url = f"https://www.alphavantage.co/query?function=TOP_GAINERS_LOSERS&apikey={alpha_vantage_key}"
    response = requests.get(url)
    if response.status_code != 200:
        raise ValueError("Error fetching top movers data.")
    data = response.json()
    snapshot['top_gainers'] = data.get('top_gainers', [])
    snapshot['top_losers'] = data.get('top_losers', [])
    snapshot['most_active'] = data.get('most_actively_traded', [])
    #Fetch Dow Jones, NasDaq, S&P500
    indices = ['^DJI', '^IXIC', '^GSPC']
    snapshot['indices'] = {}
    for sym in indices:
        url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={sym}&apikey={alpha_vantage_key}"
        response = requests.get(url)
        data = response.json()
        snapshot['indices'][sym] = data.get('Global Quote', {})
        time.sleep(0.1)
    #Fetch VIX (volatility index)
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol=^VIX&apikey={alpha_vantage_key}"
    response = requests.get(url)
    data = response.json()
    snapshot['vix'] = data.get('Global Quote', {})
    # Enhance top 3 in each category with technical indicators (RSI, MACD on 5-min interval)
    categories = ['top_gainers', 'top_losers', 'most_active']
    for category in categories:
        for i in range(min(3, len(snapshot[category]))):
            symbol = snapshot[category][i]['ticker']
            # Fetch RSI (14-period, 5-min interval)
            url = f"https://www.alphavantage.co/query?function=RSI&symbol={symbol}&interval=5min&time_period=14&series_type=close&apikey={alpha_vantage_key}"
            response = requests.get(url)
            data = response.json()
            rsi_data = data.get('Technical Analysis: RSI', {})
            latest_rsi = list(rsi_data.values())[0]['RSI'] if rsi_data else 'N/A'
            snapshot[category][i]['rsi_5min'] = latest_rsi
            time.sleep(0.1)
            # Fetch MACD (5-min interval)
            url = f"https://www.alphavantage.co/query?function=MACD&symbol={symbol}&interval=5min&series_type=close&apikey={alpha_vantage_key}"
            response = requests.get(url)
            data = response.json()
            macd_data = data.get('Technical Analysis: MACD', {})
            latest_macd = list(macd_data.values())[0] if macd_data else 'N/A'
            snapshot[category][i]['macd_5min'] = latest_macd
            time.sleep(0.1)
    #Market News Sentiment
    url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&limit=5&apikey={alpha_vantage_key}"
    response = requests.get(url)
    data = response.json()
    snapshot['market_news'] = data.get('feed', [])[:5]
    #Fetch PCE (Personal Consumption Expenditures Price Index) from FRED API (optional)
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id=PCEPI&api_key={fred_key}&file_type=json&limit=1&sort_order=desc"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        latest = data.get('observations', [{}])[0]
        snapshot['pce'] = {
                'date': latest.get('date', 'N/A'),
                'value': latest.get('value', 'N/A')
            }
    else:
            snapshot['pce'] = {'error': 'Error fetching PCE data'}
    #Compute market sentiment (bullish, bearish, or neutral)
    indices_changes = []
    for sym in indices:
        change_str = snapshot['indices'][sym].get('10. change percent', '0%')
        try:
            change = float(change_str.strip('%'))
            indices_changes.append(change)
        except ValueError:
            pass
    avg_change = sum(indices_changes) / len(indices_changes) if indices_changes else 0
    vix_value = float(snapshot['vix'].get('05. price', '0'))
    if avg_change > 0 and vix_value < 20:
        sentiment = 'bullish'
        reasoning = f"Average index change positive ({avg_change:.2f}%) and VIX low ({vix_value}). Market appears set for upward momentum."
    elif avg_change < 0 or vix_value > 30:
        sentiment = 'bearish'
        reasoning = f"Average index change negative ({avg_change:.2f}%) or VIX high ({vix_value}). Market may behave downward."
    else:
        sentiment = 'neutral'
        reasoning = f"Mixed signals: Average index change {avg_change:.2f}%, VIX at {vix_value}. Market behavior uncertain."
    snapshot['market_sentiment'] = {'sentiment': sentiment, 'reasoning': reasoning}
    
    return snapshot

In [ ]:
@tool
def stream_scan_real_time_data(interval=30):
    alpha_vantage_key = os.environ['ALPHAVANTAGE_API_KEY']
    today = datetime.now().date()
    while True: 
        try:
            #test 
            url = f"https://www.alphavantage.co/query?function=TOP_GAINERS_LOSERS&apikey={alpha_vantage_key}"
            response = requests.get(url)
            if response.status_code != 200:
                print(f"Error fetching top gainers: {response.status_code}")
                time.sleep(interval)
                continue
            data = response.json()
            gainers = data.get('top_gainers', [])
            candidates = []
            for gainer in gainers: 
                symbol = gainer.get('ticker')
                price_str = gainer.get('price', '0')
                change_str = gainer.get('change_percent', '0%')
                volume_str = gainer.get('volume', '0')

                try:
                    price = float(price_str)
                    change_percent = float(change_str.strip('%'))
                    current_volume = int(volume_str.replace(',', ''))
                except ValueError:
                    continue 

                if(change_percent >= 5 and 2 <= price <= 20 and current_volume > 500000):
                    candidates.append({
                        'symbol': symbol,
                        'price': price,
                        'change_percent': change_percent,
                        'current_volume': current_volume
                    })
                time.sleep(0.1)  # Light buffer
            # Step 2: For each candidate, check volume ratio and news
            matches = []
            for cand in candidates:
                symbol = cand['symbol']
                
                # Fetch 50-day avg volume
                url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&outputsize=compact&apikey={alpha_vantage_key}"
                response = requests.get(url)
                if response.status_code != 200:
                    time.sleep(1)
                    continue
                hist_data = response.json()
                time_series = hist_data.get('Time Series (Daily)', {})
                
                volumes = []
                for date_str, day_data in sorted(time_series.items(), key=lambda x: x[0], reverse=True)[:50]:
                    vol_str = day_data.get('5. volume', '0')
                    volumes.append(float(vol_str))
                
                if len(volumes) < 50:
                    continue  # Not enough history
                
                avg_50d_volume = sum(volumes) / len(volumes)
                volume_ratio = cand['current_volume'] / avg_50d_volume if avg_50d_volume > 0 else 0
                
                if volume_ratio < 3:
                    continue
                
                # Fetch recent news
                url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={symbol}&limit=5&apikey={alpha_vantage_key}"
                response = requests.get(url)
                news_data = response.json() if response.status_code == 200 else {}
                feed = news_data.get('feed', [])
                
                recent_news = [item for item in feed if 'time_published' in item]
                today_news = [item for item in recent_news if datetime.strptime(item['time_published'][:10], '%Y%m%d').date() == today]
                
                if not today_news:
                    continue
                
                # Match found
                match = cand.copy()
                match['avg_50d_volume'] = avg_50d_volume
                match['volume_ratio'] = volume_ratio
                match['news_items'] = today_news  # List of news dicts (title, summary, etc.)
                matches.append(match)
                
                time.sleep(1)  # Rate limit buffer per symbol (adjust for premium)
            
            # Yield matches
            for match in matches:
                yield match
            print(f"Scan complete at {datetime.now()}. Found {len(matches)} matches.")
        except Exception as e:
            print(f"Scan error: {e}")
        time.sleep(interval)

In [ ]:
@tool
def analyze_candles(open_prices, close_prices, high_prices, low_prices, threshold=0.1, support_level=None):
    """
    Analyzes a series of candles (OHLC data) to detect 'good' Doji patterns in the latest candle.
    A 'good' Doji is one where the body size is <= threshold * range, indicating indecision.
    Classifies as Neutral, Dragonfly (bullish reversal potential), or Gravestone (bearish reversal potential).
    Optionally checks if the Doji occurs near a provided support level (e.g., for stronger signal if at support).

    Args:
        open_prices (list or np.array): List of open prices (latest at end).
        close_prices (list or np.array): List of close prices (latest at end).
        high_prices (list or np.array): List of high prices (latest at end).
        low_prices (list or np.array): List of low prices (latest at end).
        threshold (float, optional): Max body-to-range ratio for Doji detection (default 0.1).
        support_level (float, optional): Support price level; if provided, checks proximity (within 1% of low).

    Returns:
        dict: {
            'is_doji': bool,
            'doji_type': str ('Neutral', 'Dragonfly', 'Gravestone', or 'None'),
            'body_size': float,
            'range': float,
            'at_support': bool (if support_level provided),
            'reasoning': str
        }
    """
    if not (len(open_prices) == len(close_prices) == len(high_prices) == len(low_prices)):
        raise ValueError("All OHLC lists must be the same length.")
    
    # Convert to arrays for ease
    opens = np.array(open_prices)
    closes = np.array(close_prices)
    highs = np.array(high_prices)
    lows = np.array(low_prices)
    
    if len(opens) == 0:
        return {'is_doji': False, 'doji_type': 'None', 'body_size': 0, 'range': 0, 'at_support': False, 'reasoning': 'No data provided.'}
    
    # Analyze latest candle
    o, c, h, l = opens[-1], closes[-1], highs[-1], lows[-1]
    body = abs(o - c)
    range_ = h - l
    if range_ == 0:
        return {'is_doji': False, 'doji_type': 'None', 'body_size': 0, 'range': 0, 'at_support': False, 'reasoning': 'Invalid candle (zero range).'}
    
    body_ratio = body / range_
    is_doji = body_ratio <= threshold
    
    if not is_doji:
        return {'is_doji': False, 'doji_type': 'None', 'body_size': body, 'range': range_, 'at_support': False, 
                'reasoning': f'Body ratio {body_ratio:.2f} > threshold {threshold}; not a Doji.'}
    
    # Classify Doji type
    upper_shadow = h - max(o, c)
    lower_shadow = min(o, c) - l
    shadow_ratio = 2  # Threshold for long shadows (adjustable if needed)
    
    if lower_shadow > shadow_ratio * body and upper_shadow < body:
        doji_type = 'Dragonfly'  # Bullish potential, especially at support
        reasoning = 'Dragonfly Doji detected (long lower shadow); potential bullish reversal.'
    elif upper_shadow > shadow_ratio * body and lower_shadow < body:
        doji_type = 'Gravestone'  # Bearish potential, especially at resistance
        reasoning = 'Gravestone Doji detected (long upper shadow); potential bearish reversal.'
    else:
        doji_type = 'Neutral'
        reasoning = 'Neutral Doji detected; market indecision.'
    
    # Check support proximity if provided
    at_support = False
    if support_level is not None:
        proximity_threshold = 0.01  # 1% of price
        if abs(l - support_level) / support_level <= proximity_threshold:
            at_support = True
            reasoning += f' Doji near support level {support_level:.2f} (low: {l:.2f}). Stronger signal.'
    return {
        'is_doji': True,
        'doji_type': doji_type,
        'body_size': body,
        'range': range_,
        'at_support': at_support,
        'reasoning': reasoning
    }


In [6]:
tools = [search, fetch_market_snapshot, stream_scan_real_time_data, analyze_candles]

In [8]:
#create model
model = init_chat_model('grok-2', model_provider='xai')

In [17]:
#memory 
memory = MemorySaver()

In [11]:
#specify role prompt
agentrole = SystemMessage(content='''You are an advanced AI agent powered by Grok from xAI, 
a real-time stock market scanner and trading decision assistant. Focus on identifying tradeable stocks via user criteria using premium APIs (yfinance, Finnhub, Alpha Vantage) for streaming data. 
Provide doji candlestick-based strategy support with buy/sell limits, stop-losses, and risk management (e.g., <2% risk per trade).
Be data-driven, relentless, and provide the utmost helpful decision to make in real time.
Principles: Prioritize live data, cross-verify APIs, quantify risks, use markdown/tables/charts. Handle errors gracefully; prompt for API keys if needed.
Workflow:
1. Onboard: Greet, gather the stock universe, criteria (e.g., volume>1M, RSI<30), doji params (thresholds, timeframe), limits, and keys.
2. Scan: Fetch/stream data with tools; output table of matches (Symbol, Price, Volume, Doji?).
3. Analyze: Detect dojis, simulate trades, recommend entries/exits with visuals.
4. Once a target stock is identified, work directly with user. Allow them to input High, Low, Open, Close values of candles. Suggest the absolute best move.
5. Monitor: Stream alerts for triggers; allow refinements.
6. Iterate: Seek feedback after outputs; refine and propose next steps.
Tools (chain as needed): get_stock_info(symbol), stream_real_time_data(api, symbols, interval), scan_stocks(criteria, universe), analyze_technical_indicators(symbol, indicators), detect_candlestick_patterns(symbol, timeframe, 'doji'), simulate_strategy(params), generate_report(format).
End responses with questions to engage. Adapt to user expertise; use step-by-step reasoning.''')

In [19]:
#create agent with params 
tradeagent = create_react_agent(model, tools, prompt=agentrole, checkpointer=memory)
config = {'configurable': {'thread_id': 'thankyou33'}}

In [20]:
print('Lets get started')
name = 'You'
userid = name + ': '
startagent = input('Confirm (Y) to begin or (N) to exit: ')
if startagent != 'Y':
    print('Cancelled Agent Request')
else:
    print('Starting, to end/cancel your agent, simply input "exit" "stop" or "cancel"')
    while True: 
        myinput = input(f"{userid}").strip()
        if myinput in ['exit', 'stop', 'cancel']:
            print('Agent Disconnected')
            break
        for step in tradeagent.stream(
            {"messages": [('user', myinput)]}, config, stream_mode='values'
        ):
            step['messages'][-1].pretty_print()

Lets get started
Cancelled Agent Request
